# Process Fugro CSV Files

This notebook converts raw Fugro CSV files into standardized per-series CSV files with consistent column names:
- First column (index): `Time` (datetime)
- Second column (data): `head` (numeric)

Output files are saved to `output_data/only_csv_fugro/`

In [2]:
# Helper functions and setup
from pathlib import Path
import re
import pandas as pd


def find_repo_root(start=Path.cwd()):
    """Return the first ancestor (including start) that contains .git or pyproject.toml."""
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / '.git').exists() or (candidate / 'pyproject.toml').exists():
            return candidate
    return p


def sanitize_for_filename(name: str) -> str:
    """
    Make a string safe for filenames on Windows/Linux/macOS by replacing
    unwanted characters with underscores and collapsing repeats.
    """
    safe = re.sub(r'[^0-9A-Za-z._-]+', '_', str(name))
    safe = safe.strip(' ._')
    return safe or "series"


repo_root = find_repo_root()
print('Repository root detected as:', repo_root)

Repository root detected as: D:\Users\jvanruitenbeek\data_validation


In [3]:
# Processing Function

In [4]:
def process_fugro_csv(input_file, repo_root, dayfirst=True, drop_all_nan=True):
    """
    Process a Fugro-style CSV (including Vista Data Vision format):

    Args:
        input_file (str or Path): Path to input Fugro CSV file
        repo_root (str or Path): Repository root path
        dayfirst (bool): Whether dates are in D-M-Y format (default: True for European format)
        drop_all_nan (bool): Skip columns that are entirely NaN (default: True)
    
    Returns:
        list: Paths to all saved CSV files
    """

    input_path = Path(input_file)
    repo_root = Path(repo_root)

    out_dir = repo_root / "output_data" / "only_csv_fugro"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Detect Vista Data Vision format
    skiprows = 0
    try:
        with open(input_path, "r", encoding="utf-8-sig", errors="replace") as f:
            first_line = f.readline().strip()
            if "Vista Data Vision" in first_line:
                skiprows = 6
                print(f"Detected Vista Data Vision file → skipping {skiprows} header rows")
    except:
        pass

    # Read raw CSV
    df = pd.read_csv(
        input_path,
        sep=",",
        header=0,
        dtype="object",
        encoding="utf-8-sig",
        engine="python",
        skiprows=skiprows,
    )

    # Normalize first column to "Time"
    first_col = str(df.columns[0]).replace("\ufeff", "").strip()
    if first_col.lower() != "time":
        df.rename(columns={df.columns[0]: "Time"}, inplace=True)

    # ------------------------------------------------------------------
    #      ROBUST TIMESTAMP PARSING WITH AUTO-DETECTION
    # ------------------------------------------------------------------
    time_raw = df["Time"].astype(str)

    # Try ISO format (YYYY-MM-DD)
    dt_iso = pd.to_datetime(time_raw, format="%Y-%m-%d %H:%M:%S", errors="coerce")

    # Try flexible dayfirst parsing
    dt_dayfirst = pd.to_datetime(time_raw, dayfirst=True, errors="coerce")

    # Pick the parsing that yields MORE valid timestamps
    if dt_iso.notna().sum() >= dt_dayfirst.notna().sum():
        df["Time"] = dt_iso
        # print("Using ISO parsing")
    else:
        df["Time"] = dt_dayfirst
        # print("Using dayfirst parsing")

    # Remove invalid timestamps & set index
    df = df.dropna(subset=["Time"]).set_index("Time")

    # ------------------------------------------------------------------
    # Save each column as its own head-series CSV
    # ------------------------------------------------------------------
    written = []
    seen_names = {}

    for col in df.columns:
        ser = pd.to_numeric(df[col], errors="coerce")

        if drop_all_nan and ser.notna().sum() == 0:
            continue

        base_name = sanitize_for_filename(col)
        count = seen_names.get(base_name, 0)
        out_name = base_name if count == 0 else f"{base_name}_{count}"
        seen_names[base_name] = count + 1

        out_df = pd.DataFrame({"head": ser})

        out_path = out_dir / f"{out_name}.csv"
        out_df.to_csv(out_path, index=True, index_label="Time")

        print(f"Saved {out_path} ({ser.notna().sum()} rows)")

        written.append(out_path)

    print(f"Done. Saved {len(written)} series to {out_dir}")
    return written

In [8]:
# List all available Fugro CSV files
fugro_dir = repo_root / 'input_data' / 'Fugro'
files = sorted(fugro_dir.glob('*.csv'))
print(f'Found {len(files)} CSV file(s) in {fugro_dir.name}:\n')
for i, f in enumerate(files, 1):
    print(f'{i:2d}. {f.name}')


file_to_process = 3

# Process a specific file
if len(files) > 0:
    print(f'\nProcessing: {files[file_to_process].name}')
    process_fugro_csv(
        input_file=files[file_to_process],
        repo_root=repo_root
    )

Found 4 CSV file(s) in Fugro:

 1. 241417_Hoorn.csv
 2. 253677_Heerhugowaard_Geavanceerd.csv
 3. 260484_Heerhugowaard_normaal.csv
 4. 263591_Hoorn_Zuiderdijk.csv

Processing: 263591_Hoorn_Zuiderdijk.csv
Detected Vista Data Vision file → skipping 6 header rows
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5029_B09PB01_-7.1_-8.1_m_NAP_avg.csv (8449 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5029_B09PB02_-2.3_-3.3_m_NAP_avg.csv (8449 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5036_B08PB01_-5.0_-6.0_m_NAP_avg.csv (8425 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5036_B08PB02_-1.4_-2.4_m_NAP_avg.csv (8425 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5055_HB02PB01_-1.5_-2.5_m_NAP_avg.csv (8430 rows)
Saved D:\Users\jvanruitenbeek\data_validation\out

C:\Users\jvanruitenbeek\AppData\Local\Temp\65\ipykernel_8832\33731997.py:57: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt_dayfirst = pd.to_datetime(time_raw, dayfirst=True, errors="coerce")


Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5095_B07PB01_-1.2_-2.2_m_NAP_avg.csv (8445 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5140_B10PB01_-0.3_-1.3_m_NAP_avg.csv (8461 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5141_B11PB01_-4.8_-5.8_m_NAP_avg.csv (8450 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5141_B11PB02_-2.4_-3.4_m_NAP_avg.csv (8450 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5153_HB03PB01_-1.9_-2.9_m_NAP_avg.csv (8449 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro\NL-263591-FB-FLB5163_B07aPB2_-5.5_-6.5_m_NAP_avg.csv (7266 rows)
Done. Saved 14 series to D:\Users\jvanruitenbeek\data_validation\output_data\only_csv_fugro
